# 03. Knapsack Family (Subset Sum Style DP)

"Choose items to reach a capacity/target" is one of the biggest DP families. Master **0/1** vs **unbounded**, and the **loop order** rules.

**Learning order:** 0/1 KNAPSACK → PARTITION → TARGET SUM → LAST STONE → UNBOUNDED → PERMUTATIONS COUNT → ONES AND ZEROES → INTERVIEW THINKING

## 1. 0/1 Knapsack

Items with `weight` and `value`, capacity `W`. Each item can be taken **at most once**. Maximize value.

2D idea: `dp[i][w]` = best value using the first `i` items with capacity `w`: `max(skip, take) = max(dp[i-1][w], dp[i-1][w - wt] + val)`.

**1D optimization:** `dp[w]`, and loop `w` from **high to low**. Going downwards guarantees `dp[w - wt]` still holds the value **before** this item (so the item is used once). Going upwards would reuse the item (that is the unbounded version).

In [1]:
def knapsack_01(weights, values, W):
    dp = [0] * (W + 1)
    for wt, val in zip(weights, values):
        for w in range(W, wt - 1, -1):               # reverse: each item used once
            dp[w] = max(dp[w], dp[w - wt] + val)
    return dp[W]

print(knapsack_01([1, 3, 4, 5], [1, 4, 5, 7], 7))   # 9

9


## 2. Partition Equal Subset Sum

Can the array be split into two subsets with equal sum? If the total is odd, no. Otherwise: can some subset sum to `total / 2`? Boolean 0/1 knapsack: `dp[s]` = can we reach sum `s`.

In [2]:
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False
    target = total // 2
    dp = [True] + [False] * target
    for x in nums:
        for s in range(target, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    return dp[target]

print(can_partition([1, 5, 11, 5]), can_partition([1, 2, 3, 5]))   # True False

True False


## 3. Target Sum (assign + or -)

Let `P` = sum of numbers with `+`, `N` = sum with `-`. `P - N = target` and `P + N = total` ⇒ `P = (total + target) / 2`. So the problem becomes: **count subsets with sum `P`** (counting version of 0/1 knapsack). Check parity and sign first.

In [3]:
def find_target_sum_ways(nums, target):
    total = sum(nums)
    if abs(target) > total or (total + target) % 2:
        return 0
    goal = (total + target) // 2
    dp = [1] + [0] * goal
    for x in nums:
        for s in range(goal, x - 1, -1):
            dp[s] += dp[s - x]
    return dp[goal]

print(find_target_sum_ways([1, 1, 1, 1, 1], 3), find_target_sum_ways([1], 1))   # 5 1

5 1


## 4. Last Stone Weight II

Smash stones; the final stone weight is minimal when we split the stones into two groups with sums as **close** as possible. Find the **largest reachable subset sum ≤ total/2**; answer = `total - 2 * best`.

In [4]:
def last_stone_weight_ii(stones):
    total = sum(stones)
    half = total // 2
    dp = [True] + [False] * half
    for x in stones:
        for s in range(half, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    best = max(s for s in range(half + 1) if dp[s])
    return total - 2 * best

print(last_stone_weight_ii([2, 7, 4, 1, 8, 1]), last_stone_weight_ii([31, 26, 33, 21, 40]))   # 1 5

1 5


## 5. Unbounded Knapsack

Each item can be used **any number of times**. Same 1D table, but loop `w` **upwards** so the current item can be used again.

In [5]:
def knapsack_unbounded(weights, values, W):
    dp = [0] * (W + 1)
    for wt, val in zip(weights, values):
        for w in range(wt, W + 1):                   # forward: reuse allowed
            dp[w] = max(dp[w], dp[w - wt] + val)
    return dp[W]

print(knapsack_unbounded([1, 3, 4], [15, 50, 60], 8))   # 130

130


## 6. Combination Sum IV (count **permutations**)

`[1, 1, 2]` and `[2, 1, 1]` count as different. Put the **target loop on the outside** and the numbers inside: each `dp[t]` sums over **every possible last number**, so order matters.

In [6]:
def combination_sum4(nums, target):
    dp = [1] + [0] * target
    for t in range(1, target + 1):                   # target outer => permutations
        for x in nums:
            if x <= t:
                dp[t] += dp[t - x]
    return dp[target]

print(combination_sum4([1, 2, 3], 4))   # 7

7


## 7. Ones and Zeroes (2D capacity)

Pick the maximum number of binary strings using at most `m` zeros and `n` ones. A knapsack with **two capacities**: `dp[z][o]`. Each string has a cost `(zeros, ones)` and value `1`. Loop both capacities downwards.

In [7]:
def find_max_form(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        zeros, ones = s.count("0"), s.count("1")
        for z in range(m, zeros - 1, -1):
            for o in range(n, ones - 1, -1):
                dp[z][o] = max(dp[z][o], dp[z - zeros][o - ones] + 1)
    return dp[m][n]

print(find_max_form(["10", "0001", "111001", "1", "0"], 5, 3))   # 4

4


## 8. The loop-order cheat sheet

| Problem type | Items loop | Capacity loop | Result |
|---|---|---|---|
| 0/1 (each once) | outer | **downwards** | max value / count subsets |
| Unbounded, **combinations** | outer | **upwards** | Coin Change II |
| Unbounded, **permutations** | inner | outer (capacity) | Combination Sum IV |
| Min items to reach target | any | upwards | Coin Change |

## 9. Interview thinking

**Signals:** "subset with sum", "split into two equal parts", "assign +/-", "capacity", "at most m zeros and n ones", "use each item once / any number of times".

**Say:** "This is a 0/1 knapsack: `dp[s]` is whether (or how many ways) sum `s` is reachable. I iterate `s` downwards so each item is used once. O(n × target) time, O(target) space."

**Common mistakes:** wrong loop direction (reuse vs once), wrong loop nesting (combinations vs permutations), forgetting the parity check (total odd), negative targets, not handling `dp[0] = 1` / `True`.

## Quick Revision

- 0/1: capacity **downwards**. Unbounded: **upwards**.
- Combinations: items outer. Permutations: capacity outer.
- Equal partition = subset sum `total/2`.
- Target sum: `P = (total + target)/2`, count subsets.
- Stones II: `total - 2 * best_subset_sum ≤ total/2`.